<a href="https://colab.research.google.com/github/amrtaher1234/Iqraeli/blob/main/Iqraeli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.4/225.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.5.0
    Uninstalling typing_extensions-4.5.0:
      Successfully uninstalled typing_extensions-4.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.
tensorflow-probability 0.22.0 requires typing-extensions<4.6.0, but you have typing-extensions 4.9.0 which is incompatible.


In [ ]:
from openai import OpenAI

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import json

In [ ]:
# Utilities
def load_json_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return json.load(file)

def write_json_data(data, pathname):
     with open(pathname, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

In [ ]:
from google.colab import userdata
client = OpenAI(api_key=userdata.get('OPEN_AI_KEY'))


In [55]:
def create_embeddings(objects, pathname):
    arabic_texts = [obj['merged_tafseer_text'] for obj in objects]

    embeddings_items = client.embeddings.create(model='text-embedding-ada-002', input=arabic_texts).data
    embeddings = []

    for item in  embeddings_items:
        embeddings.append(item.embedding)


    for obj, embedding in zip(objects, embeddings):
        obj['embedding'] = embedding
        # obj['embedding_text'] = surah_only_embedding

    with open(pathname, 'w', encoding='utf-8') as file:
        json.dump(objects, file, ensure_ascii=False, indent=4)



In [36]:
quran_with_tafseer = load_json_data('./raw/quran-with-tafseer.json')

In [ ]:
create_embeddings(objects=quran_with_tafseer[0:2000], pathname='./embeddings/tafseer-with-quran-embeddings.json')
create_embeddings(objects=quran_with_tafseer[2000:4000], pathname='./embeddings/tafseer-with-quran-embeddings2.json')
create_embeddings(objects=quran_with_tafseer[4000:6000], pathname='./embeddings/tafseer-with-quran-embeddings3.json')
create_embeddings(objects=quran_with_tafseer[6000:], pathname='./embeddings/tafseer-with-quran-embeddings4.json')

In [54]:
# used for preprocessing and combining quran data, use the `quran-embeddings.json` directly

file_paths = [
    './embeddings/tafseer-with-quran-embeddings.json',
    './embeddings/tafseer-with-quran-embeddings-2.json',
    './embeddings/tafseer-with-quran-embeddings-3.json',
    './embeddings/tafseer-with-quran-embeddings-4.json'
]

combined_data = []
for path in file_paths:
    data = load_json_data(path)
    combined_data.extend(data)

write_json_data(combined_data, './embeddings/quran-embeddings.json')

In [49]:

def find_closest_5(objects, query):
    query_embedding = client.embeddings.create(model='text-embedding-ada-002', input=query).data[0].embedding
    top_objects = []

    for obj in objects:
        similarity = cosine_similarity([query_embedding], [obj['embedding']])[0][0]
        top_objects.append((obj, similarity))
        top_objects.sort(key=lambda x: x[1], reverse=True)
        top_objects = top_objects[:5]

    [print(obj[1]) for obj in top_objects]
    return [obj[0] for obj in top_objects]


In [42]:
embeddings_data = load_json_data('./embeddings/quran-embeddings.json')



In [57]:
top_5_data = find_closest_5(embeddings_data, 'الجنة ونعيم الجنة')

# Remove the 'embedding' field from each object and print
for data in top_5_data:
    print(data['surah_name_english'])
    print(data['aya_number'])
    print('-----')

6236
0.8696352270300682
0.8685465918807352
0.8681588925049692
0.8658198438121953
0.8617296479160115
An-Nas
6
-----
As-Sajda
13
-----
At-Tur
17
-----
Al-Infitar
13
-----
An-Nazi'at
41
-----
